# Fire Event AI Confidence Model Pipeline
This notebook implements the end-to-end AI training pipeline for the Fire Event AI Confidence Model.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
import json
import os
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, precision_recall_fscore_support
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

2026/09/26 01:13:40 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at D:\Projects\Blockchain-Based-Automated-Fire-Insurance-Claim-System\.venv\Lib\site-packages\mlflow\assistant\skills\instrumenting-with-mlflow-tracing\SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


## Phase 1: Data preprocessing
- Load CSV
- Convert `flame_detected` to numeric, parse `timestamp`
- Handle outliers

In [2]:
data_path = '../../data/raw/fire_sensor_training_data.csv'
df = pd.read_csv(data_path)

# Convert types
df['flame_detected'] = df['flame_detected'].astype(int)
df['timestamp'] = pd.to_datetime(df['timestamp'])

# Display initial stats
print(f"Dataset shape: {df.shape}")
print(df.isnull().sum())

# Handle outliers: clip extreme values for temperature and smoke_level to physical limits
df['temperature'] = df['temperature'].clip(lower=-20, upper=200)
df['smoke_level'] = df['smoke_level'].clip(lower=0, upper=1023)

# Data Quality Report
quality_report = {
    "total_rows": len(df),
    "class_distribution": df['label'].value_counts().to_dict(),
    "missing_values": df.isnull().sum().to_dict(),
    "temperature_stats": df['temperature'].describe().to_dict(),
    "smoke_level_stats": df['smoke_level'].describe().to_dict()
}

os.makedirs('../../ai/reports', exist_ok=True)
with open('../../ai/reports/data_quality_report.md', 'w') as f:
    f.write("# Data Quality Report\n")
    f.write(f"**Total rows:** {quality_report['total_rows']}\n\n")
    f.write("**Class Distribution:**\n")
    for k, v in quality_report['class_distribution'].items():
        f.write(f"- {k}: {v}\n")
    f.write("\n**Missing Values:** None\n")
    f.write(f"\n**Temperature Stats:** Mean {quality_report['temperature_stats']['mean']:.2f}, Max {quality_report['temperature_stats']['max']:.2f}\n")

Dataset shape: (1800, 11)
event_id            0
device_id           0
property_id         0
temperature         0
smoke_level         0
flame_detected      0
duration_seconds    0
latitude            0
longitude           0
timestamp           0
label               0
dtype: int64


## Phase 2: Feature Engineering
- `temp_smoke_interaction`: Interaction term
- `is_high_risk`: Categorical flag

In [3]:
df['temp_smoke_interaction'] = df['temperature'] * df['smoke_level']
df['is_high_risk'] = ((df['temperature'] > 60) & (df['flame_detected'] == 1)).astype(int)

# Features to use for modeling
numeric_features = ['temperature', 'smoke_level', 'duration_seconds', 'temp_smoke_interaction']
categorical_features = ['flame_detected', 'is_high_risk']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', 'passthrough', categorical_features)
    ])

X = df[numeric_features + categorical_features]
y = df['label']

## Phase 3: Train/Test Split
We use a 70/15/15 stratified split.

In [4]:
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.15/0.85, stratify=y_temp, random_state=42)

print("Train balance:")
print(y_train.value_counts(normalize=True))
print("Val balance:")
print(y_val.value_counts(normalize=True))
print("Test balance:")
print(y_test.value_counts(normalize=True))

Train balance:


label
NORMAL            0.500000
POSSIBLE_FIRE     0.166667
CONFIRMED_FIRE    0.166667
ANOMALY           0.166667
Name: proportion, dtype: float64
Val balance:
label
NORMAL            0.500000
ANOMALY           0.166667
POSSIBLE_FIRE     0.166667
CONFIRMED_FIRE    0.166667
Name: proportion, dtype: float64
Test balance:
label
NORMAL            0.500000
ANOMALY           0.166667
POSSIBLE_FIRE     0.166667
CONFIRMED_FIRE    0.166667
Name: proportion, dtype: float64


## Phase 4 & 5: Model training with hyperparameter tuning & Cross-Validation
We train:
1. Isolation Forest on NORMAL data only.
2. Random Forest on all data.

In [5]:
# Setting up MLflow
mlflow.set_tracking_uri("sqlite:///../../ai/mlruns/mlflow.db")
mlflow.set_experiment("fire-event-ai-confidence")

# --- Candidate 1: Isolation Forest ---
print("Training Isolation Forest...")
X_train_normal = X_train[y_train == 'NORMAL']

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
if_params = {'contamination': [0.01, 0.05, 0.1], 'n_estimators': [100, 200]}

best_if_score = -1
best_if_model = None
best_if_param = None

for cont in if_params['contamination']:
    for n_est in if_params['n_estimators']:
        with mlflow.start_run(run_name=f"IsolationForest_c{cont}_n{n_est}"):
            mlflow.log_param("model", "IsolationForest")
            mlflow.log_param("contamination", cont)
            mlflow.log_param("n_estimators", n_est)
            
            y_val_if = np.where(y_val == 'NORMAL', 1, -1)
            
            if_pipeline = Pipeline([
                ('preprocessor', preprocessor),
                ('if', IsolationForest(contamination=cont, n_estimators=n_est, random_state=42))
            ])
            
            if_pipeline.fit(X_train_normal)
            preds = if_pipeline.predict(X_val)
            
            precision, recall, f1, _ = precision_recall_fscore_support(y_val_if, preds, average='macro')
            mlflow.log_metric("val_f1_macro", f1)
            
            if f1 > best_if_score:
                best_if_score = f1
                best_if_model = if_pipeline
                best_if_param = {"contamination": cont, "n_estimators": n_est}

print(f"Best IF params: {best_if_param} with F1 macro {best_if_score:.4f}")

# --- Candidate 2: Random Forest ---
print("Training Random Forest...")
rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('rf', RandomForestClassifier(random_state=42))
])

rf_params = {
    'rf__n_estimators': [50, 100],
    'rf__max_depth': [3, 5, 10]
}

rf_search = GridSearchCV(rf_pipeline, rf_params, cv=cv, scoring='f1_macro')

with mlflow.start_run(run_name="RandomForest_GridSearch"):
    mlflow.sklearn.autolog()
    rf_search.fit(X_train, y_train)

best_rf_model = rf_search.best_estimator_
best_rf_score = rf_search.best_score_
print(f"Best RF params: {rf_search.best_params_} with CV F1 macro {best_rf_score:.4f}")

2026/09/26 01:13:59 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/09/26 01:13:59 INFO mlflow.store.db.utils: Updating database tables


2026/09/26 01:14:02 INFO mlflow.tracking.fluent: Experiment with name 'fire-event-ai-confidence' does not exist. Creating a new experiment.


Training Isolation Forest...


Best IF params: {'contamination': 0.05, 'n_estimators': 100} with F1 macro 0.9666
Training Random Forest...


...

2026/09/26 01:14:21 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


2026/09/26 01:14:22 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


2026/09/26 01:14:26 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


2026/09/26 01:14:26 INFO mlflow.sklearn.utils: Logging the 5 best runs, one run will be omitted.


Best RF params: {'rf__max_depth': 10, 'rf__n_estimators': 100} with CV F1 macro 0.9346


## Phase 6: Experiment tracking with MLflow
Both model runs were logged to MLflow above.

## Phase 7: Final model selection and save

In [6]:
print("--- Isolation Forest Validation ---")
if_test_preds = best_if_model.predict(X_test)
if_test_mapped = np.where(if_test_preds == 1, 'NORMAL', 'ANOMALY')
y_test_binary = np.where(y_test == 'NORMAL', 'NORMAL', 'ANOMALY')
print(classification_report(y_test_binary, if_test_mapped))

print("--- Random Forest Validation ---")
rf_test_preds = best_rf_model.predict(X_test)
print(classification_report(y_test, rf_test_preds))

import joblib
best_model = best_rf_model

os.makedirs('../../ai/model', exist_ok=True)
joblib.dump(best_model, '../../ai/model/best_model.pkl')
joblib.dump(preprocessor, '../../ai/model/preprocessing_pipeline.pkl')

metadata = {
    "model_type": "RandomForestClassifier",
    "hyperparameters": rf_search.best_params_,
    "features": numeric_features + categorical_features,
    "test_f1_macro": float(precision_recall_fscore_support(y_test, rf_test_preds, average='macro')[2])
}
with open('../../ai/model/model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=4)

print("Saved final model artifacts.")

--- Isolation Forest Validation ---
              precision    recall  f1-score   support

     ANOMALY       0.96      0.99      0.97       135
      NORMAL       0.99      0.96      0.97       135

    accuracy                           0.97       270
   macro avg       0.97      0.97      0.97       270
weighted avg       0.97      0.97      0.97       270

--- Random Forest Validation ---
                precision    recall  f1-score   support

       ANOMALY       0.89      0.91      0.90        45
CONFIRMED_FIRE       1.00      1.00      1.00        45
        NORMAL       1.00      0.99      1.00       135
 POSSIBLE_FIRE       0.91      0.91      0.91        45

      accuracy                           0.97       270
     macro avg       0.95      0.95      0.95       270
  weighted avg       0.97      0.97      0.97       270

Saved final model artifacts.
